# Causal-asymmetry metric panel — Metric 0 (PPL) + Metrics 1–3 (bits)

Reproduces the asymmetry figure but in the clean form: **Metric 0 (standard PPL)** in its
own panel (informational only), and **Metrics 1, 2, 3 together** on a shared bit axis.

This notebook *reuses the tooling already implemented in the repo* — it does not reimplement
any metric. It only adds a new plotting cell. Mapping:

| Metric | Reused function | Unit |
|---|---|---|
| 0 — standard PPL (informational) | `compute_standard_ppl` → `perplexity_ind_model` | perplexity |
| 1 — hard-label CE (primary)      | `compute_ce` → `perplexity_calculation`         | bits |
| 2 — soft-label CE                | `compute_ind_ce` → `perplexity_ind_CE`          | bits |
| 3 — mean stepwise KL             | `stepwise_kl_coin`                              | bits |
| H∞ — entropy-rate floor          | `entropy_rate_coin`                             | bits |

**Where to run it:** place this notebook in the repo root (`LLM_final_version/`) so the
project modules import and `results/models/` is visible. Metrics 2 and 3 are coin-only
(the flower process has no closed-form conditional), so this panel targets the coin experiments.


In [ ]:
import os
import numpy as np
import torch
import matplotlib

# Reuse the project's implemented tooling.
# NOTE: importing LLM_asymmetry_testing runs matplotlib.use("Agg") and sets usetex=True
# at import time; we override both right after so figures render inline.
from LLM_asymmetry_testing import (
    load_model,            # build OneHotDecoder + load .pt weights
    entropy_rate_coin,     # H_inf for the coin process
    stepwise_kl_coin,      # Metric 3
    compute_standard_ppl,  # Metric 0  -> perplexity_ind_model
    compute_ce,            # Metric 1  -> perplexity_calculation
    compute_ind_ce,        # Metric 2  -> perplexity_ind_CE
    CFG,                   # default config (d_model, max_len, n_layers, max_batches, ...)
)
from Data_generation import CoinDataset, coin_generation
from Training_model import _loader

import matplotlib.pyplot as plt
%matplotlib inline   # overrides the Agg backend set on import above

# Flip USETEX to True to match the paper's Computer Modern font (needs a working LaTeX install).
# All plot strings below are mathtext-compatible, so they render in either mode.
USETEX = False
plt.rcParams.update({
    "text.usetex": USETEX,
    "font.family": "serif",
    "axes.titlesize": 11,
    "axes.labelsize": 11,
    "figure.dpi": 120,
})
print("torch", torch.__version__, "| cwd:", os.getcwd())

In [ ]:
# ---- paths & experiment registry ------------------------------------------
MODELS_DIR = "results/models"
OUT_DIR    = "results/asymmetry_test"     # figures saved here, per experiment

# Coin experiments that have trained FW/BW weights in results/models/.
# (p, q) must match the values the models were trained with.
COIN_EXPERIMENTS = {
    "exp1_coin_p03_q04":   dict(p=0.3, q=0.4),
    "exp1_2_coin_p04_q08": dict(p=0.4, q=0.8),
    "exp1_2_coin_p01_q09": dict(p=0.1, q=0.9),
}

# Evaluation data scale (matches LLM_asymmetry_testing defaults).
# Reduce num_samples / set MAX_BATCHES for a quick preview; Metric 3 is the slow one.
NUM_SAMPLES = 500
SEQ_LEN     = 2000
BATCH_SIZE  = 32
NUM_TOKEN   = 3
MAX_BATCHES = CFG.get("max_batches")   # None = use all batches

TAG = "exp1_coin_p03_q04"              # <-- choose which experiment to plot
print("available:", list(COIN_EXPERIMENTS))

In [ ]:
def compute_all_metrics(tag, p, q, *, seq_len=SEQ_LEN, num_samples=NUM_SAMPLES,
                        batch_size=BATCH_SIZE, num_token=NUM_TOKEN,
                        max_batches=MAX_BATCHES, models_dir=MODELS_DIR, cfg=CFG):
    """
    Load FW/BW models for `tag` and compute all four metrics + H_inf on a freshly
    generated ground-truth coin loader. Mirrors the computation block of eval_coin,
    reusing the imported functions (no metric is reimplemented here).
    """
    fw_path = os.path.join(models_dir, f"{tag}_fw.pt")
    bw_path = os.path.join(models_dir, f"{tag}_bw.pt")
    for pth in (fw_path, bw_path):
        if not os.path.exists(pth):
            raise FileNotFoundError(pth)

    model_fw = load_model(fw_path, num_token, cfg, mode="forward")
    model_bw = load_model(bw_path, num_token, cfg, mode="backward")

    # Same ground-truth loader for both models (the principled comparison).
    data, _ = coin_generation(num_samples=num_samples, seq_len=seq_len, p=p, q=q)
    loader  = _loader(CoinDataset(data, seq_len=seq_len), batch_size)

    h_inf = entropy_rate_coin(p, q)

    # Metric 0 — standard autoregressive PPL on each model's own sequence (informational)
    ppl0_fw = compute_standard_ppl(model_fw, len_seq=seq_len)
    ppl0_bw = compute_standard_ppl(model_bw, len_seq=seq_len)

    # Metric 1 — hard-label CE on the ground-truth loader (primary signal)
    _, ce1_fw = compute_ce(model_fw, loader, max_batches)
    _, ce1_bw = compute_ce(model_bw, loader, max_batches)

    # Metric 2 — soft-label CE vs the analytic HMM conditional (ground-truth loader)
    _, ce2_fw = compute_ind_ce(model_fw, loader, p, q, cfg)
    _, ce2_bw = compute_ind_ce(model_bw, loader, p, q, cfg)

    # Metric 3 — mean stepwise KL on the ground-truth loader
    kl3_fw, _, _ = stepwise_kl_coin(model_fw, loader, p, q, num_token, max_batches)
    kl3_bw, _, _ = stepwise_kl_coin(model_bw, loader, p, q, num_token, max_batches)

    return dict(tag=tag, p=p, q=q, h_inf=h_inf,
                ppl0_fw=ppl0_fw, ppl0_bw=ppl0_bw,
                ce1_fw=ce1_fw,   ce1_bw=ce1_bw,
                ce2_fw=ce2_fw,   ce2_bw=ce2_bw,
                kl3_fw=kl3_fw,   kl3_bw=kl3_bw)


def print_summary(M):
    d1 = M["ce1_bw"] - M["ce1_fw"]
    d2 = M["ce2_bw"] - M["ce2_fw"]
    d3 = M["kl3_bw"] - M["kl3_fw"]
    print(f"=== {M['tag']}  (p={M['p']}, q={M['q']}) ===")
    print(f"H_inf (entropy-rate floor)      = {M['h_inf']:.4f} bits\n")
    print(f"[M0] standard PPL    FW / BW    = {M['ppl0_fw']:.4f} / {M['ppl0_bw']:.4f}  (informational)")
    print(f"[M1] hard-label CE   FW / BW    = {M['ce1_fw']:.4f} / {M['ce1_bw']:.4f} bits   d = {d1:+.4f}")
    print(f"[M2] soft-label CE   FW / BW    = {M['ce2_fw']:.4f} / {M['ce2_bw']:.4f} bits   d = {d2:+.4f}")
    print(f"[M3] mean KL         FW / BW    = {M['kl3_fw']:.4f} / {M['kl3_bw']:.4f} bits   d = {d3:+.4f}\n")
    print("consistency checks:")
    print(f"  M1 ~= M2 ?            FW: {M['ce1_fw']:.4f} vs {M['ce2_fw']:.4f}"
          f"   BW: {M['ce1_bw']:.4f} vs {M['ce2_bw']:.4f}")
    print(f"  M2 ~= H_inf + M3 ?    FW: {M['ce2_fw']:.4f} vs {M['h_inf']+M['kl3_fw']:.4f}"
          f"   BW: {M['ce2_bw']:.4f} vs {M['h_inf']+M['kl3_bw']:.4f}")

In [ ]:
params = COIN_EXPERIMENTS[TAG]
M = compute_all_metrics(TAG, params["p"], params["q"])
print_summary(M)

In [ ]:
FW_C, BW_C = "#4c72b0", "#dd8452"
FW_LBL, BW_LBL = r"$\leftarrow$ Forward", r"Backward $\rightarrow$"


def plot_metric_panel(M, out_path=None, usetex=None):
    """1x2 figure: Metric 0 (PPL, informational) | Metrics 1-3 (bits, asymmetry signal)."""
    if usetex is None:
        usetex = plt.rcParams["text.usetex"]
    tag_disp = M["tag"].replace("_", r"\_") if usetex else M["tag"]

    fig, (axL, axR) = plt.subplots(
        1, 2, figsize=(13, 5.5), gridspec_kw={"width_ratios": [1.0, 2.2]})

    # ---- LEFT: Metric 0 — standard PPL (informational) ----------------------
    v0 = [M["ppl0_fw"], M["ppl0_bw"]]
    b0 = axL.bar([FW_LBL, BW_LBL], v0, color=[FW_C, BW_C],
                 alpha=0.85, edgecolor="k", width=0.5)
    axL.bar_label(b0, fmt="%.4f", padding=3, fontsize=9)
    href = 2 ** M["h_inf"]
    axL.axhline(href, color="crimson", ls="--", lw=1.5,
                label=rf"$2^{{H_\infty}} = {href:.4f}$")
    axL.legend(fontsize=8, loc="lower right")
    axL.set_ylabel(r"Perplexity $2^{\mathcal{L}}$")
    axL.set_ylim(0, max(v0 + [href]) * 1.18)
    axL.set_title(r"\textit{Metric 0}: standard PPL (informational)" if usetex
                  else "Metric 0: standard PPL (informational)")
    axL.grid(True, alpha=0.3, axis="y")
    axL.text(0.5, 0.94, rf"$\Delta = {v0[1]-v0[0]:+.4f}$",
             transform=axL.transAxes, ha="center", fontsize=9)

    # ---- RIGHT: Metrics 1, 2, 3 — bits --------------------------------------
    groups = ["Metric 1\n(hard-label CE)", "Metric 2\n(soft-label CE)", "Metric 3\n(mean KL)"]
    fw = [M["ce1_fw"], M["ce2_fw"], M["kl3_fw"]]
    bw = [M["ce1_bw"], M["ce2_bw"], M["kl3_bw"]]
    x, w = np.arange(3), 0.38
    bF = axR.bar(x - w/2, fw, w, color=FW_C, alpha=0.85, edgecolor="k", label=FW_LBL)
    bB = axR.bar(x + w/2, bw, w, color=BW_C, alpha=0.85, edgecolor="k", label=BW_LBL)
    axR.bar_label(bF, fmt="%.3f", padding=2, fontsize=8)
    axR.bar_label(bB, fmt="%.3f", padding=2, fontsize=8)
    axR.axhline(M["h_inf"], color="crimson", ls="--", lw=1.5,
                label=rf"$H_\infty = {M['h_inf']:.4f}$ (floor: M1, M2)")
    top = max(fw + bw)
    for xi, (f, b) in enumerate(zip(fw, bw)):
        axR.text(xi, max(f, b) + 0.06 * top, rf"$\Delta = {b-f:+.3f}$",
                 ha="center", fontsize=8)
    axR.set_xticks(x); axR.set_xticklabels(groups, fontsize=9)
    axR.set_ylabel("bits / token")
    axR.set_ylim(0, top * 1.28)
    axR.set_title(r"\textit{Metrics 1--3}: asymmetry signal (bits)" if usetex
                  else "Metrics 1-3: asymmetry signal (bits)")
    axR.legend(fontsize=8, loc="upper left", ncol=1)
    axR.grid(True, alpha=0.3, axis="y")

    suptitle = (rf"\textbf{{{tag_disp}}} --- causal asymmetry "
                r"($\mathcal{L}_\mathrm{BW} - \mathcal{L}_\mathrm{FW}$)") if usetex else \
               (rf"{tag_disp} — causal asymmetry "
                r"($\mathcal{L}_\mathrm{BW} - \mathcal{L}_\mathrm{FW}$)")
    fig.suptitle(suptitle, fontsize=12)
    fig.tight_layout()
    if out_path:
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        fig.savefig(out_path, dpi=200, bbox_inches="tight")
        print("saved:", out_path)
    plt.show()
    return fig

In [ ]:
out_path = os.path.join(OUT_DIR, TAG, f"{TAG}_metric_panel.png")
_ = plot_metric_panel(M, out_path=out_path)

## Batch: every coin experiment

Loops over all coin experiments with trained weights and writes one panel each.
Skips any experiment whose `.pt` files are not present in `results/models/`.


In [ ]:
all_metrics = {}
for tag, prm in COIN_EXPERIMENTS.items():
    try:
        Mi = compute_all_metrics(tag, prm["p"], prm["q"])
    except FileNotFoundError as e:
        print(f"skip {tag}: missing weights ({e})")
        continue
    all_metrics[tag] = Mi
    print_summary(Mi)
    plot_metric_panel(Mi, out_path=os.path.join(OUT_DIR, tag, f"{tag}_metric_panel.png"))
    print()